# Notebook 18 detached workflow
Purpose: run TRAIN diagnostics and eight paired full-JEPA training/readouts with one original12-hour clock. Inputs: frozen18operator/proof and Food101. Outputs: checkpointed diagnostics,16trained arms,certified probes,sealed primary results and portable report. Runtime: about6–10hours, hard12-hour cap. Checkpoints: checkpoints/<RUN_ID>/. Resume: sameRUN_ID and unchangedsources via Notebook18launch-learned-kernel-study; no silent restart or timing reset.

In [ ]:
from pathlib import Path
import copy,json,os,shlex,signal,subprocess,time,traceback
import nbformat
_SELF=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/18_learned_kernel/workflow.ipynb'))
_space={'__file__':str(_SELF.with_name('common.ipynb'))}
for _cell in nbformat.read(_SELF.with_name('common.ipynb'),4).cells:
    if _cell.cell_type=='code'and'ahr-export'in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_SELF.with_name('common.ipynb')),'exec'),_space)
C=type('Common',(),_space)
MAIN='notebooks/18_Learned_Kernel_JEPA_Diagnostic_and_Learning.ipynb'
TERMINAL={'COMPLETE','UNKNOWN','FAILED','VALIDATION_COMPLETE'}
OPTIONS={'rank':64,'feature_seed':180064,'local_radius':1,'spatial_mode':'disabled','feature_temperature_min':.25,'feature_temperature_max':4.,'feature_temperature_initial':1.,'gate_initial':.5}
MAXIMUM_SECONDS=43200.
EVALUATION_RESERVE=10800.

def artifact_path(root,run_id):
    if not isinstance(run_id,str)or not run_id or any(not(c.isalnum()or c in'-_')for c in run_id):raise ValueError('Safe stable RUN_ID required')
    return Path(root).resolve()/'artifacts/notebook_18'/run_id

def session_names(run_id):
    prefix='fmm18-'+run_id;return {k:prefix+suffix for k,suffix in [('coordinator',''),('guard','-guard'),('worker0','-worker0'),('worker1','-worker1')]}

def consumer_hash(root):return C.digest([''.join(c['source'])for c in json.loads((Path(root)/MAIN).read_text())['cells']if c['cell_type']=='code'])

def status(artifact,state,phase,**details):
    value={'status':state,'phase':phase,'updated_unix':time.time(),**details};C.atomic_json(Path(artifact)/'study_status.json',value);print(state,phase,flush=True);return value

def report(root,artifact,result,diagnostic=None,development=None,training=None):
    return C.load(root,'experiments/18_learned_kernel/report.ipynb','fmm18_report').render(root,artifact,result,diagnostic,development,training)

def check_registration(root,artifact):
    req=C.read(Path(artifact)/'launch_registration.json');clock=C.read(Path(artifact)/'clock.json')
    if req['sources']!=C.source_hashes(root)or req['consumer_hash']!=consumer_hash(root):raise ValueError('Frozen18 source/frontdoor changed; exact resume refused')
    if clock!=req['clock']or C.file_record(Path(artifact)/'clock.json')!=req['clock_record']:raise ValueError('Original12-hour clock changed')
    if clock['maximum_seconds']!=MAXIMUM_SECONDS or abs(clock['deadline_unix']-clock['started_unix']-MAXIMUM_SECONDS)>1e-5:raise ValueError('Original deadline must never reset or extend')
    for key in ('operator_gate','preflight'):
        if C.file_record(req[key]['path'])!=req[key]:raise ValueError('Launch evidence changed:'+key)
    return req,clock

def available_devices(root):
    B=C.load(root,'experiments/17_fixed_kernel/benchmark.ipynb','fmm18_availability');devices=[];observations={}
    for line in subprocess.check_output(['nvidia-smi','--query-gpu=index','--format=csv,noheader'],text=True).splitlines():
        device='cuda:'+str(int(line.strip()))
        try:
            observation=B._exclusive_gpu(device);observations[device]=observation
            if observation.get('foreign_idle_context_pids'):
                observations[device]={**observation,'status':'UNAVAILABLE_FOREIGN_RESIDENT','eligible':False,'selection_reason':'Foreign resident context excluded even when momentarily idle; protects against intermittent competing compute'}
            else:devices.append(device)
        except RuntimeError as exc:observations[device]={'status':'UNAVAILABLE','reason':repr(exc)}
    if not devices:raise RuntimeError('No observed available GPU; existing jobs are preserved')
    return devices,observations

def prepare_launch(root,run_id):
    root=Path(root);R=C.runner(root);proof_artifact=artifact_path(root,run_id+'-launch-preflight-'+C.digest(C.source_hashes(root))[:10])
    test_path=root/'artifacts/notebook_18_validation/operator/validation_receipt.json'
    if not test_path.exists():raise ValueError('Run tests/18_Operator_Validation.ipynb before launch')
    operator=C.read(test_path)
    if operator.get('status')!='PASS'or operator.get('actual_cuda')!='PASS'or operator.get('operator_export_sha256')!=C.export_hash(root/'experiments/18_learned_kernel/operator.ipynb'):raise ValueError('Current CPU/CUDA own-kernel implementation proof required')
    proof=R.actual_preflight(root,proof_artifact,OPTIONS,256,4,device='cuda:0')
    if proof.get('status')!='PASS'or not proof.get('passed')or proof['source_hashes']!=C.source_hashes(root):raise ValueError('Actual256px16/8 full-state checkpoint/resume proof failed')
    operator_path=proof_artifact/'operator_numerical_gate.json'
    gate={'status':'PASS','passed':True,'source_hashes':C.source_hashes(root),'operator_source_sha256':operator['operator_export_sha256'],
        'receipt':C.file_record(test_path),'check_count':operator['check_count'],'actual_cuda':True,
        'evidence_role':'Own-kernel implementation numerical correctness only; no teacher-fidelity, efficiency or accuracy claim'}
    C.immutable(operator_path,gate)
    return C.file_record(operator_path),C.file_record(proof_artifact/'actual_preflight_result.json')

def run_validation(root,run_id='learned-kernel-frontdoor-validation'):
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    result={'status':'VALIDATION_COMPLETE','fixture_source':'SOFTWARE_PRESENTATION_ONLY; NO_SCIENTIFIC_TRAINING','official_test':'CLOSED','run_id':run_id}
    report(root,artifact,result);status(artifact,'VALIDATION_COMPLETE','software_presentation');return result

def launch_study(root,run_id='learned-kernel-001',profile=None):
    root=Path(root).resolve();C.bootstrap(root);profile=profile or os.environ.get('FMM18_PROFILE','full')
    if profile=='validation':return run_validation(root,run_id)
    if profile!='full':raise ValueError('Full or validation profile required')
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    from fmm_jepa.food101_training import run_writer
    with run_writer(artifact/'launch_writer'):
        path=artifact/'launch_registration.json'
        if path.exists():req,clock=check_registration(root,artifact)
        else:
            if (artifact/'clock.json').exists():raise ValueError('Orphan original clock requires inspection; never restart')
            operator,proof=prepare_launch(root,run_id);devices,observations=available_devices(root)
            started=time.time();clock={'started_unix':started,'deadline_unix':started+MAXIMUM_SECONDS,'maximum_seconds':MAXIMUM_SECONDS,
                'scope':'Complete diagnostic, data/model preflight,16arms,initial/finalDEVprobes, sealed officialreadout,checkpointIO','restart_policy':'No deadline reset or extension'}
            C.immutable(artifact/'clock.json',clock)
            req={'schema':'notebook18-launch/v1','run_id':run_id,'sources':C.source_hashes(root),'consumer_hash':consumer_hash(root),'clock':clock,'clock_record':C.file_record(artifact/'clock.json'),
                'operator_gate':operator,'preflight':proof,'options':OPTIONS,'devices':devices,'device_observations':observations,'input_workers':4,
                'seeds':list(range(20262101,20262109)),'active_wall_seconds_per_arm':1800.,'evaluation_reserve_seconds':EVALUATION_RESERVE,'gpu_workers':len(devices),'device_policy':'Exclude every foreign resident GPU context, including momentarily idle ones; one available worker keeps all eight pairs sequentially without changing per-model allowance',
                'official_test':'CLOSED','primary':'One fixed learnedencoder versusViT, eightpairedseeds, equalactivewall, balanced5050officialsubset,0.5pp+paired95%CI>0',
                'diagnostic_policy':'TRAIN-only frozenViT diagnostic is explanatory, not a promotion gate',
                'training_gate':'Own-kernel numerical validity and actual full-state resume only; no teacherfidelity/speed/development-quality selection',
                'additional_online_parameters':98328,'parameter_matching':'Sharedoriginalweights/predictor paired; candidate has98328 extraonlineparameters, not a same-parameter-count claim'}
            C.immutable(path,req)
        existing=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
        if existing.get('status')in TERMINAL:return {**existing,'report':str(artifact/'report.html'),'resume':'Terminal receipt retained; changed design requires a new run ID'}
        C.check_deadline(clock['deadline_unix']);sessions=session_names(run_id)
        if not (artifact/'report.html').exists():report(root,artifact,{'status':'RUNNING','phase':'launch','clock':clock,'official_test':'CLOSED','run_id':run_id})
        guard=write_operator_notebook(root,artifact/'operators/guard.ipynb','run_guard',{'root':str(root),'run_id':run_id})
        coordinator=write_operator_notebook(root,artifact/'operators/coordinator.ipynb','run_coordinator',{'root':str(root),'run_id':run_id})
        a=spawn(root,guard,sessions['guard'],artifact/'guard.log');b=spawn(root,coordinator,sessions['coordinator'],artifact/'coordinator.log')
        receipt={'status':'DETACHED_PIPELINE_STARTED','run_id':run_id,'clock':clock,'sessions':sessions,'guard':a,'coordinator':b,
            'artifact':str(artifact),'report':str(artifact/'report.html'),'attach':'tmux attach -t '+sessions['coordinator'],'detach':'Ctrl-b then d',
            'resume':'Fresh ahr-go kernel, sameRUN_ID, Notebook18RunAll, unchangedsources/config/clock. IN_FLIGHT timers stayUNKNOWN; never forgive unmeasuredtime.'}
        C.atomic_json(artifact/'launch_receipt.json',receipt);return receipt

def gate_receipts(req):
    return [{'role':role,'status':'PASS','passed':True,'evidence':req[key],
        'evidence_role':'NUMERICAL_OR_RESUME_ONLY; NO_ACCURACY_OR_SPEED_SELECTION'}for role,key in [('operator','operator_gate'),('exact_resume','preflight')]]

def registered_workers(req):return tuple('worker'+str(i)for i in range(len(req['devices'])))

def worker_folder(artifact,worker):return Path(artifact)/'learning/workers'/worker

def run_worker(root,run_id,worker):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact);folder=worker_folder(artifact,worker);R=C.runner(root)
    plan=C.read(folder/'plan.json');reserve=req['evaluation_reserve_seconds']
    try:
        status(folder,'RUNNING','data_and_exact_resume');C.environment(plan['arms'][0]['config'])
        bundle=R.data_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=False)
        training=R.train_worker(root,folder,plan,bundle,clock['deadline_unix']-reserve)
        if training.get('status')!='COMPLETE':status(folder,'UNKNOWN','training_fairness');return training
        status(folder,'RUNNING','initial_and_final_converged_readouts')
        evaluation=R.evaluate_worker(root,folder,plan,bundle,training,clock['deadline_unix']-3600)
        state='COMPLETE'if evaluation.get('status')=='COMPLETE'else'UNKNOWN';status(folder,state,'finished')
        return {'status':state,'training':str(folder/'training_result.json'),'evaluation':str(folder/'evaluation_result.json')}
    except BaseException as exc:
        status(folder,'UNKNOWN','exception',reason=repr(exc),traceback=traceback.format_exc());raise

def launch_wait_workers(root,artifact,req,plan):
    R=C.runner(root);sessions=session_names(req['run_id']);clock=req['clock'];workers=registered_workers(req)
    for worker in workers:
        folder=worker_folder(artifact,worker);folder.mkdir(parents=True,exist_ok=True);child=R.worker_plan(plan,worker);child['stage_id']='learning-'+worker
        C.immutable(folder/'plan.json',child)
        state=C.read(folder/'study_status.json')if(folder/'study_status.json').exists()else{}
        if state.get('status')=='COMPLETE':continue
        if state.get('status')in('UNKNOWN','FAILED'):raise ValueError('Prior worker incomplete; preserve UNKNOWN timing evidence')
        notebook=write_operator_notebook(root,folder/'worker.ipynb','run_worker',{'root':str(root),'run_id':req['run_id'],'worker':worker})
        spawn(root,notebook,sessions[worker],folder/'worker.log')
    started=time.time()
    while True:
        C.check_deadline(clock['deadline_unix'])
        states={w:C.read(worker_folder(artifact,w)/'study_status.json')if(worker_folder(artifact,w)/'study_status.json').exists()else{'status':'STARTING'}for w in workers}
        if any(s['status']in('UNKNOWN','FAILED')for s in states.values()):raise RuntimeError('Worker incomplete:'+str(states))
        if all(s['status']=='COMPLETE'for s in states.values()):break
        if time.time()-started>90 and any(states[w]['status']!='COMPLETE'and not session_exists(sessions[w])for w in workers):raise RuntimeError('Owned worker session disappeared; preserve exact timing state')
        C.atomic_json(Path(artifact)/'coordinator_heartbeat.json',{'updated_unix':time.time(),'phase':'learning_and_readouts','worker_states':states,'deadline_unix':clock['deadline_unix']});time.sleep(10)
    training={'status':'COMPLETE','fair_wall_budget_verified':True,'arms':{},'gate_receipts':gate_receipts(req)};rows=[];initial=[]
    for worker in workers:
        folder=worker_folder(artifact,worker);t=C.read(folder/'training_result.json');e=C.read(folder/'evaluation_result.json')
        if t['status']!='COMPLETE'or not t['fair_wall_budget_verified']or e['status']!='COMPLETE':raise ValueError('Full paired training and certified readouts required')
        training['arms'].update(t['arms']);rows.extend(e['rows']);initial.extend(e['initial_rows']);terminate_session_tree(sessions[worker])
    values=list(training['arms'].values());tol=max(plan['wall_tolerance_seconds'],plan['active_wall_seconds_per_arm']*plan['wall_tolerance_fraction'])
    if len(values)!=16 or max(v['active_wall_seconds']for v in values)-min(v['active_wall_seconds']for v in values)>tol:raise ValueError('All16arms equalactivewallfairness required')
    evaluation={'status':'COMPLETE','rows':rows,'initial_rows':initial,'official_test':'CLOSED','source_hashes':req['sources']}
    C.atomic_json(Path(artifact)/'learning/training_result.json',training);C.atomic_json(Path(artifact)/'learning/evaluation_result.json',evaluation)
    return training,evaluation

def run_coordinator(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact);diagnostic=None;development=None;training=None
    from fmm_jepa.food101_training import run_writer
    try:
        with run_writer(artifact/'coordinator_writer'):
            R=C.runner(root);F=C.load(root,'experiments/18_learned_kernel/confirmation.ipynb','fmm18_confirmation');folder=artifact/'learning';folder.mkdir(parents=True,exist_ok=True)
            plan_path=folder/'plan.json';plan=C.read(plan_path)if plan_path.exists()else R.make_screen_plan(root,run_id,req['options'],req['devices'],clock['deadline_unix'],4,gate_receipts(req))
            C.immutable(plan_path,plan);bundle=R.data_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=True)
            # Freeze hypothesis, allseedpairs and finalmembershipBEFORE scientifictraining.
            F.register_confirmation(root,folder,plan,bundle)
            status(artifact,'RUNNING','frozen_training_token_diagnostic')
            D=C.load(root,'experiments/18_learned_kernel/diagnostics.ipynb','fmm18_diagnostics')
            activation=artifact/'diagnostic_activation.json'
            if activation.exists():diagnostic_clock=C.read(activation)
            else:
                diagnostic_clock={'started_unix':time.time(),'deadline_unix':min(clock['deadline_unix']-EVALUATION_RESERVE,time.time()+900),'source_hashes':req['sources'],'maximum_seconds':900}
                C.immutable(activation,diagnostic_clock)
            if diagnostic_clock['source_hashes']!=req['sources']or diagnostic_clock['deadline_unix']>clock['deadline_unix']-EVALUATION_RESERVE:raise ValueError('Original diagnostic activation changed')
            diagnostic=D.run_diagnostic(root,artifact/'diagnostic',deadline=diagnostic_clock['deadline_unix'],profile='full')
            report(root,artifact,{'status':'RUNNING','phase':'paired_full_jepa_learning','official_test':'CLOSED','run_id':run_id},diagnostic)
            status(artifact,'RUNNING','eight_fresh_pairs_full_jepa_learning')
            training,evaluation=launch_wait_workers(root,artifact,req,plan)
            # Development analysis issecondary andnever selects official models/opening.
            development=F.analyze_development(plan,evaluation);C.atomic_json(folder/'development_result.json',development)
            status(artifact,'RUNNING','sealed_official_readout');report(root,artifact,{'status':'RUNNING','phase':'sealed_official_readout','official_test':'CLOSED','run_id':run_id},diagnostic,development,training)
            result=F.seal_and_score(root,folder,plan,bundle,training,{'model_builder':R.builder(root,req['options']),'rows':evaluation['rows']},clock['deadline_unix'])
            result.update(run_id=run_id,source_hashes=req['sources'],diagnostic_status=diagnostic['status'],additional_online_parameters=98328,
                analysis_scope='One predeclared candidate,eightfreshpairedseeds,equal1800activewallseconds; DEVresultssecondary',development=C.file_record(folder/'development_result.json'))
            report(root,artifact,result,diagnostic,development,training);status(artifact,result['status'],'finished',decision=result.get('decision'),report=str(artifact/'report.html'));return result
    except BaseException as exc:
        stopped={w:terminate_session_tree(session_names(run_id)[w])for w in('worker0','worker1')}
        result={'status':'UNKNOWN','reason':repr(exc),'traceback':traceback.format_exc(),'official_test':'CLOSED_UNLESS_VALID_EXISTING_SEAL','stopped_workers':stopped,'run_id':run_id}
        report(root,artifact,result,diagnostic,development,training);status(artifact,'UNKNOWN','cutoff_or_integrity_failure',reason=repr(exc));return result

def run_guard(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);sessions=session_names(run_id);last_review=0.;missing_since=None
    try:
        while True:
            req,clock=check_registration(root,artifact);existing=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
            if existing.get('status')in TERMINAL:return {'status':'GUARD_COMPLETE','study_status':existing['status']}
            now=time.time()
            if not session_exists(sessions['coordinator']):
                if missing_since is None:missing_since=now
                elif now-missing_since>90:raise RuntimeError('Coordinator absent beyondstartupgrace')
            else:missing_since=None
            if now>=clock['deadline_unix']:raise TimeoutError('Original12-hour deadline exhausted')
            C.atomic_json(artifact/'guard_heartbeat.json',{'status':'HEALTHY','updated_unix':now,'deadline_unix':clock['deadline_unix'],'sessions':{k:session_exists(v)for k,v in sessions.items()}})
            if now-last_review>=3600:
                R,_,_=C.modules(root);audits=[]
                for manifest in(root/'checkpoints'/run_id).glob('stages/*/arms/*/manifest.json'):
                    current=validate_manifest_rotation(R,manifest.parent);audits.append({'path':str(manifest.parent),'successful_updates':current['successful_updates'],'latest_valid':True})
                C.atomic_json(artifact/'hourly_review.json',{'status':'PASS','updated_unix':now,'audits':audits,'source_and_original_clock_valid':True});last_review=now
            time.sleep(10)
    except BaseException as exc:
        stopped={w:terminate_session_tree(sessions[w])for w in('coordinator','worker0','worker1')}
        result={'status':'UNKNOWN','reason':repr(exc),'stopped_project_processes':stopped,'official_test':'CLOSED_UNLESS_VALID_EXISTING_SEAL','run_id':run_id}
        report(root,artifact,result);status(artifact,'UNKNOWN','guard_deadline_or_integrity_failure',reason=repr(exc));return result


In [ ]:
def session_exists(name):return subprocess.run(['tmux','has-session','-t','='+name],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0


def terminate_session_tree(session):
    """Stop only processes descending from this exact project-owned tmux session."""
    if not session_exists(session):return []
    roots={int(x)for x in subprocess.check_output(['tmux','list-panes','-t','='+session,'-F','#{pane_pid}'],text=True).split()}
    edges=[tuple(map(int,line.split()))for line in subprocess.check_output(['ps','-eo','pid=,ppid='],text=True).splitlines()];owned=set(roots)
    while True:
        expanded=owned|{pid for pid,parent in edges if parent in owned}
        if expanded==owned:break
        owned=expanded
    for pid in sorted(owned-roots,reverse=True)+sorted(roots):
        try:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    subprocess.run(['tmux','kill-session','-t','='+session],capture_output=True);return sorted(owned)


def validate_manifest_rotation(runner,directory):
    for attempt in range(4):
        try:return runner.validate_training_manifest(directory)
        except (FileNotFoundError,ValueError):
            if attempt==3:raise
            time.sleep(.25)


def write_operator_notebook(root,path,function,arguments):
    code="from pathlib import Path\nimport json,nbformat,types,os\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\nROOT=Path("+repr(str(Path(root).resolve()))+")\nP=ROOT/'experiments/18_learned_kernel/workflow.ipynb'\nW=types.ModuleType('fmm18_detached_workflow');W.__file__=str(P)\nfor cell in nbformat.read(P,4).cells:\n    if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(P),'exec'),W.__dict__)\nRESULT=W."+function+"(**json.loads("+repr(json.dumps(arguments))+"))\nprint(json.dumps(RESULT,indent=2,default=str))\n"
    nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Detached18 '+function+'\nGenerated operator entrypoint. Frozen parent source, original deadline and single-writer checkpoints remain authoritative. Resume through Notebook18 with the same RUN_ID.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR FMM-JEPA','language':'python'}})
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        old=nbformat.read(path,4)
        if [c.source for c in old.cells if c.cell_type=='code']!=[c.source for c in nb.cells if c.cell_type=='code']:raise ValueError('Detached entrypoint code changed')
    else:nbformat.write(nb,path)
    return path


def spawn(root,path,session,log):
    if session_exists(session):return {'status':'EXISTING_SESSION','session':session}
    jupyter=Path(root).parent/'.venv/bin/jupyter';command=' '.join(shlex.quote(str(x))for x in[jupyter,'nbconvert','--to','notebook','--execute','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',path])+' >> '+shlex.quote(str(log))+' 2>&1'
    env='CUBLAS_WORKSPACE_CONFIG=:4096:8 OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 '
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),env+command],check=True)
    return {'status':'STARTED','session':session,'operator_notebook':str(path),'log':str(log)}
